In [1]:
import boto3
from pprint import pprint
import pandas as pd
from io import StringIO
import json

In [2]:
# Get list of all objects from the bucket
s3_client = boto3.client('s3')
bucket_name = 'data605-final-project'

paginator = s3_client.get_paginator('list_objects_v2')

all_objects = []

for page in paginator.paginate(Bucket=bucket_name):
    all_objects.extend(page.get('Contents', []))

In [3]:
# Load all objects using get_object method
all_jsons = []

for obj in all_objects:
    if not obj['Key'].endswith('.json'):
        continue

    response = s3_client.get_object(
        Bucket=bucket_name,
        Key=obj['Key']
    )

    # 
    res = json.loads(response['Body'].read().decode('utf-8'))
    all_jsons.append(res)

In [4]:
# Verify that there are 3105 JSON files
print(len(all_jsons))

3105


In [87]:
df_list = []
df = pd.DataFrame(all_jsons)

In [88]:
df

,name,date,tech_self_score,strengths,weaknesses,self_development,geo_flex,financial_support_self,result,course_interest
0,Stillmann Castano,22/08/2019,"{'C#': 6, 'Java': 5, 'R': 2, 'JavaScript': 2}",[Charisma],"[Distracted, Impulsive, Introverted]",Yes,Yes,Yes,Pass,Business
1,Hilary Willmore,01/08/2019,"{'Python': 1, 'C#': 4, 'Java': 2, 'C++': 4}","[Patient, Curious, Problem Solving]","[Overbearing, Chatty, Indifferent]",No,Yes,Yes,Fail,Data
2,Efrem Whipple,22/08/2019,"{'Ruby': 4, 'C++': 4}","[Courteous, Independent, Patient]","[Introverted, Impulsive, Anxious]",Yes,Yes,Yes,Pass,Business
3,Sydel Fenne,28/08/2019,"{'Java': 3, 'SPSS': 4}",[Passionate],"[Perfectionist, Sensitive]",Yes,Yes,Yes,Pass,Data
4,Michel Lebarree,07/08/2019,"{'Python': 3, 'Java': 4, 'Ruby': 1, 'R': 2, 'P...",[Versatile],"[Controlling, Perfectionist, Chatty]",Yes,Yes,Yes,Pass,Engineering
...,...,...,...,...,...,...,...,...,...,...
3100,Jacky Reilingen,04/04/2019,"{'C#': 2, 'Java': 6, 'R': 1, 'SPSS': 4}",[Versatile],"[Indifferent, Intolerant, Introverted]",Yes,No,Yes,Fail,Business
3101,Phillis Lyfield,10/04/2019,"{'C#': 4, 'Java': 4, 'Ruby': 4, 'PHP': 1}","[Organisation, Independent, Determined]","[Sensitive, Overbearing, Impatient]",Yes,Yes,Yes,Pass,Engineering
3102,Celle Barlas,16/04/2019,"{'R': 2, 'C++': 1, 'JavaScript': 4}",[Problem Solving],[Critical],Yes,Yes,Yes,Pass,Engineering
3103,Scott Duny,11/04/2019,{'Ruby': 3},"[Reliable, Perfectionism, Problem Solving]",[Stubborn],Yes,Yes,Yes,Pass,Data


In [89]:
# Drop columns with multiple values in one cell
columns_to_drop = ['tech_self_score', 'strengths', 'weaknesses']
df = df.drop(columns=columns_to_drop)
df

,name,date,self_development,geo_flex,financial_support_self,result,course_interest
0,Stillmann Castano,22/08/2019,Yes,Yes,Yes,Pass,Business
1,Hilary Willmore,01/08/2019,No,Yes,Yes,Fail,Data
2,Efrem Whipple,22/08/2019,Yes,Yes,Yes,Pass,Business
3,Sydel Fenne,28/08/2019,Yes,Yes,Yes,Pass,Data
4,Michel Lebarree,07/08/2019,Yes,Yes,Yes,Pass,Engineering
...,...,...,...,...,...,...,...
3100,Jacky Reilingen,04/04/2019,Yes,No,Yes,Fail,Business
3101,Phillis Lyfield,10/04/2019,Yes,Yes,Yes,Pass,Engineering
3102,Celle Barlas,16/04/2019,Yes,Yes,Yes,Pass,Engineering
3103,Scott Duny,11/04/2019,Yes,Yes,Yes,Pass,Data


In [90]:
# Change yes and no to booleans
df['self_development'] = df['self_development'].map({
    'Yes': True,
    'No': False
})

df['geo_flex'] = df['geo_flex'].map({
    'Yes': True,
    'No': False
})

df['financial_support_self'] = df['financial_support_self'].map({
    'Yes': True,
    'No': False
})

In [91]:
df

,name,date,self_development,geo_flex,financial_support_self,result,course_interest
0,Stillmann Castano,22/08/2019,True,True,True,Pass,Business
1,Hilary Willmore,01/08/2019,False,True,True,Fail,Data
2,Efrem Whipple,22/08/2019,True,True,True,Pass,Business
3,Sydel Fenne,28/08/2019,True,True,True,Pass,Data
4,Michel Lebarree,07/08/2019,True,True,True,Pass,Engineering
...,...,...,...,...,...,...,...
3100,Jacky Reilingen,04/04/2019,True,False,True,Fail,Business
3101,Phillis Lyfield,10/04/2019,True,True,True,Pass,Engineering
3102,Celle Barlas,16/04/2019,True,True,True,Pass,Engineering
3103,Scott Duny,11/04/2019,True,True,True,Pass,Data


In [92]:
# Change result column to pass with a Boolean value 
df = df.rename(columns={'result': 'pass'})

In [93]:
df['pass'] = df['pass'].map({
    'Pass': True,
    'Fail': False
})
df

,name,date,self_development,geo_flex,financial_support_self,pass,course_interest
0,Stillmann Castano,22/08/2019,True,True,True,True,Business
1,Hilary Willmore,01/08/2019,False,True,True,False,Data
2,Efrem Whipple,22/08/2019,True,True,True,True,Business
3,Sydel Fenne,28/08/2019,True,True,True,True,Data
4,Michel Lebarree,07/08/2019,True,True,True,True,Engineering
...,...,...,...,...,...,...,...
3100,Jacky Reilingen,04/04/2019,True,False,True,False,Business
3101,Phillis Lyfield,10/04/2019,True,True,True,True,Engineering
3102,Celle Barlas,16/04/2019,True,True,True,True,Engineering
3103,Scott Duny,11/04/2019,True,True,True,True,Data


In [94]:
df.head()

,name,date,self_development,geo_flex,financial_support_self,pass,course_interest
0,Stillmann Castano,22/08/2019,True,True,True,True,Business
1,Hilary Willmore,01/08/2019,False,True,True,False,Data
2,Efrem Whipple,22/08/2019,True,True,True,True,Business
3,Sydel Fenne,28/08/2019,True,True,True,True,Data
4,Michel Lebarree,07/08/2019,True,True,True,True,Engineering


In [95]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3105 entries, 0 to 3104
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   name                    3105 non-null   str  
 1   date                    3105 non-null   str  
 2   self_development        3105 non-null   bool 
 3   geo_flex                3105 non-null   bool 
 4   financial_support_self  3105 non-null   bool 
 5   pass                    3105 non-null   bool 
 6   course_interest         3105 non-null   str  
dtypes: bool(4), str(3)
memory usage: 85.0 KB


In [96]:
df.duplicated().sum()

np.int64(32)

In [97]:
# Check the duplicated rows
df[df.duplicated(keep=False)]

,name,date,self_development,geo_flex,financial_support_self,pass,course_interest
58,Pamelina Itzkovsky,13/08/2019,True,True,True,True,Engineering
59,Pamelina Itzkovsky,13/08/2019,True,True,True,True,Engineering
153,Rafi Andrzejczak,07/08/2019,True,False,True,False,Business
154,Rafi Andrzejczak,07/08/2019,True,False,True,False,Business
210,Redford Bubbings,15/08/2019,False,True,True,False,Engineering
...,...,...,...,...,...,...,...
2765,Trent Southwick,12/09/2019,True,True,True,True,Engineering
2970,Pren Abramovicz,18/04/2019,True,True,True,True,Business
2971,Pren Abramovicz,18/04/2019,True,True,True,True,Business
3040,Daniela Raffan,24/04/2019,True,True,True,True,Engineering


In [98]:
# Drop duplicates
df = df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

In [102]:
# Clean the formatting of dates
df['date'] = df['date'].str.replace('//', '/', regex=False)

In [106]:
# Convert date column to datetime
df['date'] = pd.to_datetime(df['date'], dayfirst=True)
df.info()

<class 'pandas.DataFrame'>
Index: 3073 entries, 0 to 3104
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   name                    3073 non-null   str           
 1   date                    3073 non-null   datetime64[us]
 2   self_development        3073 non-null   bool          
 3   geo_flex                3073 non-null   bool          
 4   financial_support_self  3073 non-null   bool          
 5   pass                    3073 non-null   bool          
 6   course_interest         3073 non-null   str           
dtypes: bool(4), datetime64[us](1), str(2)
memory usage: 108.0 KB


In [123]:
# Check if anyone has missing tech self scores
for person in all_jsons:
    if 'tech_self_score' not in person:
        print(person['name'])

Haslett Steet
Wesley Tolerton
Alvie Bleackly
Andy Lown
Alister Bru
Morgen Lilywhite
Markos Walter
Keen Tripony
Amil Lidden
Rancell Lill
Lannie Minister
Joseito Duiged
Rachele Bracknell
Dermot Gunthorp
Julie Condliffe
Chrissy Sate
Farlee Petren
Adolpho Litterick
Tiphanie Selway
Deeyn Klimkin
Uriel Lamplough
Essy Haggus
Cirilo Croy
Rollo Quarry
Farah Ely
Rooney Sheldrick
Rooney Sheldrick
Kerby Tranfield
Gui Graine
Fran Gebbie
Stephan Bahls
Farrell Normington
Rudy Davidde
Jacquelin Clampin
Vannie Hounsome
Loree Hinkens
Francisco Tebboth
Herby Tomaini
Arne Dybell
Lannie Mcdonell
Jori Riseley
Michell Fishlock
Tildie Joselevitz
Doe Eisold
Dulcine Duffil
Chrissy Bartholin
Davidde Tuohy
Kippy Dallon
Myrta Tissell
Dasi Goldston
Claiborne Jesper
Beltran Jolly
Elizabet Sanbroke
Erna Milne
Mimi Beamand


In [125]:
tech_list = []
for person in all_jsons:
    name = person['name']
    date = person['date']
    
    if 'tech_self_score' not in person:
        continue
        
    for skill, self_score in person['tech_self_score'].items():
        tech_list.append([name, date, skill, self_score])

tech_df = pd.DataFrame(tech_list, columns=['name', 'date', 'skill', 'self_score'])

In [126]:
tech_df

,name,date,skill,self_score
0,Stillmann Castano,22/08/2019,C#,6
1,Stillmann Castano,22/08/2019,Java,5
2,Stillmann Castano,22/08/2019,R,2
3,Stillmann Castano,22/08/2019,JavaScript,2
4,Hilary Willmore,01/08/2019,Python,1
...,...,...,...,...
9765,Scott Duny,11/04/2019,Ruby,3
9766,Boycey Matushenko,25/04/2019,Python,2
9767,Boycey Matushenko,25/04/2019,C#,3
9768,Boycey Matushenko,25/04/2019,Java,3


In [127]:
tech_df

<class 'pandas.DataFrame'>
RangeIndex: 9770 entries, 0 to 9769
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   name        9770 non-null   str  
 1   date        9770 non-null   str  
 2   skill       9770 non-null   str  
 3   self_score  9770 non-null   int64
dtypes: int64(1), str(3)
memory usage: 305.4 KB


In [129]:
# Clean the formatting of dates
tech_df['date'] = tech_df['date'].str.replace('//', '/', regex=False)

In [130]:
# Convert date column to datetime
tech_df['date'] = pd.to_datetime(tech_df['date'], dayfirst=True)
tech_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9770 entries, 0 to 9769
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   name        9770 non-null   str           
 1   date        9770 non-null   datetime64[us]
 2   skill       9770 non-null   str           
 3   self_score  9770 non-null   int64         
dtypes: datetime64[us](1), int64(1), str(2)
memory usage: 305.4 KB


In [131]:
tech_df

,name,date,skill,self_score
0,Stillmann Castano,2019-08-22,C#,6
1,Stillmann Castano,2019-08-22,Java,5
2,Stillmann Castano,2019-08-22,R,2
3,Stillmann Castano,2019-08-22,JavaScript,2
4,Hilary Willmore,2019-08-01,Python,1
...,...,...,...,...
9765,Scott Duny,2019-04-11,Ruby,3
9766,Boycey Matushenko,2019-04-25,Python,2
9767,Boycey Matushenko,2019-04-25,C#,3
9768,Boycey Matushenko,2019-04-25,Java,3


In [135]:
strength_list = []
for person in all_jsons:
    name = person['name']
    date = person['date']
    
    if 'strengths' not in person:
        continue
        
    for strength in person['strengths']:
        strength_list.append([name, date, strength])

In [137]:
strength_df = pd.DataFrame(strength_list,columns=['name', 'date', 'strength'])
strength_df

,name,date,strength
0,Stillmann Castano,22/08/2019,Charisma
1,Hilary Willmore,01/08/2019,Patient
2,Hilary Willmore,01/08/2019,Curious
3,Hilary Willmore,01/08/2019,Problem Solving
4,Efrem Whipple,22/08/2019,Courteous
...,...,...,...
6248,Celle Barlas,16/04/2019,Problem Solving
6249,Scott Duny,11/04/2019,Reliable
6250,Scott Duny,11/04/2019,Perfectionism
6251,Scott Duny,11/04/2019,Problem Solving


In [138]:
weakness_list = []
for person in all_jsons:
    name = person['name']
    date = person['date']
    
    if 'weaknesses' not in person:
        continue
        
    for weakness in person['weaknesses']:
        weakness_list.append([name, date, weakness])

In [139]:
weakness_df = pd.DataFrame(weakness_list,columns=['name', 'date', 'weakness'])
weakness_df

,name,date,weakness
0,Stillmann Castano,22/08/2019,Distracted
1,Stillmann Castano,22/08/2019,Impulsive
2,Stillmann Castano,22/08/2019,Introverted
3,Hilary Willmore,01/08/2019,Overbearing
4,Hilary Willmore,01/08/2019,Chatty
...,...,...,...
6159,Phillis Lyfield,10/04/2019,Impatient
6160,Celle Barlas,16/04/2019,Critical
6161,Scott Duny,11/04/2019,Stubborn
6162,Boycey Matushenko,25/04/2019,Controlling


In [141]:
# Clean the formatting of dates
weakness_df['date'] = weakness_df['date'].str.replace('//', '/', regex=False)

In [142]:
# Convert date column to datetime
tech_df['date'] = pd.to_datetime(tech_df['date'], dayfirst=True)
tech_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9770 entries, 0 to 9769
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   name        9770 non-null   str           
 1   date        9770 non-null   datetime64[us]
 2   skill       9770 non-null   str           
 3   self_score  9770 non-null   int64         
dtypes: datetime64[us](1), int64(1), str(2)
memory usage: 305.4 KB
